In [5]:
import os
import torch
import scanpy as sc
import numpy as np
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [6]:
class CLIPDataset(Dataset):
    def __init__(self, cell_embedding_path, image_embedding_path):
        self.cell_embedding_path = Path(cell_embedding_path)
        self.image_embedding_path = Path(image_embedding_path)
        
        self.data = []
        
        for cell_file in self.cell_embedding_path.glob('*.h5ad'):
            adata = sc.read_h5ad(cell_file)
            cell_embeddings = adata.obsm['emb']
            
            image_folder = self.image_embedding_path / cell_file.stem
            
            for idx, cell_emb in zip(adata.obs.index, cell_embeddings):
                image_file = image_folder / f"cell_{idx}.pt"
                if image_file.exists():
                    self.data.append((cell_emb, image_file))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        cell_embedding, image_file = self.data[idx]

        cell_embedding = torch.tensor(cell_embedding, dtype=torch.float32)

        image_embedding = torch.load(image_file)
        
        return cell_embedding, image_embedding



In [4]:

cell_embedding_path = 'data/cell_embeddings'
image_embedding_path = 'path/to/image_embeddings'


In [ ]:

dataset = CLIPDataset(cell_embedding_path, image_embedding_path)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)


In [29]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.layer_norm1 = nn.LayerNorm(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.layer_norm1(self.fc1(x))
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [30]:


class CLIPModel(nn.Module):
    def __init__(self, image_dim, gene_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super(CLIPModel, self).__init__()
        self.image_mlp = MLP(image_dim, hidden_dim, output_dim, dropout_rate)
        self.gene_mlp = MLP(gene_dim, hidden_dim, output_dim, dropout_rate)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, image_features, gene_features):
        image_embeddings = self.image_mlp(image_features)
        gene_embeddings = self.gene_mlp(gene_features)

        # Normalize embeddings
        image_embeddings = F.normalize(image_embeddings, dim=-1)
        gene_embeddings = F.normalize(gene_embeddings, dim=-1)

        # Scaled pairwise cosine similarities
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_embeddings @ gene_embeddings.t()
        logits_per_gene = logits_per_image.t()

        return logits_per_image, logits_per_gene


In [31]:
class InfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07):
        super(InfoNCELoss, self).__init__()
        self.temperature = temperature

    def forward(self, logits_per_image, logits_per_gene):
        batch_size = logits_per_image.size(0)

        targets = torch.arange(batch_size).long().to(logits_per_image.device)

        loss_img = F.cross_entropy(logits_per_image / self.temperature, targets)
        loss_gene = F.cross_entropy(logits_per_gene / self.temperature, targets)

        return (loss_img + loss_gene) / 2


In [32]:
def train_clip(model, dataloader, optimizer, device, temperature=0.07):
    model.train()
    criterion = InfoNCELoss(temperature) 

    total_loss = 0
    for image_batch, gene_batch in dataloader:
        image_batch = image_batch.to(device)
        gene_batch = gene_batch.to(device)

        optimizer.zero_grad()

        logits_per_image, logits_per_gene = model(image_batch, gene_batch)

        # 使用InfoNCELoss计算损失
        loss = criterion(logits_per_image, logits_per_gene)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)
    return average_loss


In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [34]:

image_dim = 512 
gene_dim = 1000 
hidden_dim = 256
output_dim = 128

model = CLIPModel(image_dim, gene_dim, hidden_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



In [35]:

num_epochs = 10
for epoch in range(num_epochs):
    loss = train_clip(model, dataloader, optimizer, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

Epoch 1/10, Loss: 4.8922
Epoch 2/10, Loss: 4.1604
Epoch 3/10, Loss: 4.1572
Epoch 4/10, Loss: 4.1544
Epoch 5/10, Loss: 4.1509
Epoch 6/10, Loss: 4.1474
Epoch 7/10, Loss: 4.1440
Epoch 8/10, Loss: 4.1408
Epoch 9/10, Loss: 4.1379
Epoch 10/10, Loss: 4.1346


In [8]:
class SimulatedDataset(Dataset):
    def __init__(self, num_samples, image_dim, gene_dim):
        self.num_samples = num_samples
        self.image_embeddings = torch.randn(num_samples, image_dim)
        self.gene_expressions = torch.randn(num_samples, gene_dim)
        
        # 添加一些相关性
        noise = torch.randn(num_samples, min(image_dim, gene_dim)) * 0.1
        self.image_embeddings[:, :min(image_dim, gene_dim)] += noise
        self.gene_expressions[:, :min(image_dim, gene_dim)] += noise

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.image_embeddings[idx], self.gene_expressions[idx]

def custom_collate_fn(batch):
    image_embeddings = []
    gene_expressions = []
    
    for image_emb, gene_exp in batch:
        image_embeddings.append(image_emb)
        gene_expressions.append(gene_exp)
    
    image_embeddings = torch.stack(image_embeddings)
    gene_expressions = torch.stack(gene_expressions)
    
    return image_embeddings, gene_expressions


In [20]:
num_samples = 1000000
image_dim = 512
gene_dim = 1000
batch_size = 64


In [21]:

dataset = SimulatedDataset(num_samples, image_dim, gene_dim)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)


In [22]:
for image_batch, gene_batch in dataloader:
    print(f"Image batch shape: {image_batch.shape}")
    print(f"Gene batch shape: {gene_batch.shape}")
    break



Image batch shape: torch.Size([64, 512])
Gene batch shape: torch.Size([64, 1000])


In [23]:

hidden_dim = 1024
output_dim = 128



In [24]:


if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print(f"Using the first two GPUs out of {torch.cuda.device_count()} available GPUs")
    device = torch.device("cuda:0")  
    use_multi_gpu = True
else:
    print("Using CPU or single GPU")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    use_multi_gpu = False
    


Using the first two GPUs out of 4 available GPUs


In [25]:
model = CLIPModel(image_dim, gene_dim, hidden_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



In [26]:

num_epochs = 10
for epoch in range(num_epochs):
    loss = train_clip(model, dataloader, optimizer, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

Epoch 1/10, Loss: 4.1538
Epoch 2/10, Loss: 4.1953
Epoch 3/10, Loss: 4.0433
Epoch 4/10, Loss: 3.9617
Epoch 5/10, Loss: 3.7742
Epoch 6/10, Loss: 3.6940
Epoch 7/10, Loss: 3.6197
Epoch 8/10, Loss: 3.6648
Epoch 9/10, Loss: 3.4602
Epoch 10/10, Loss: 3.5638
